In [ ]:
!pip install sentence-transformers
!pip install indic-nlp-library  # Install the library

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 48.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlink

In [ ]:
from IPython import get_ipython
from IPython.display import display
import pandas as pd
import re
import unicodedata
from sentence_transformers import SentenceTransformer, util
from google.colab import files
from indicnlp.tokenize import sentence_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from functools import lru_cache

In [ ]:
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/combined_data.csv')

In [ ]:
def clean_nepali_text(text):
    # Remove HTML tags using a more robust regex
    text = re.sub(r'<.*?>', '', text)

    # Remove punctuation and special characters (except for Devanagari script)
    # Keep common Nepali punctuation: ।, ?
    text = re.sub(r'[^\u0900-\u097F\s।?]', '', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Unicode normalization
    text = unicodedata.normalize('NFC', text)

    # 6. Handle Nepali numbers (optional - convert to Arabic numerals)
    nepali_numbers = '०१२३४५६७८९'
    arabic_numbers = '0123456789'
    translation_table = str.maketrans(nepali_numbers, arabic_numbers)
    text = text.translate(translation_table)

    return text

In [ ]:
#Create the stop word removal function (same as before)
def remove_stopwords(text):
    #Load stop words from the text file
    with open('stopwords.txt', 'r', encoding='utf-8') as f:  # Assuming UTF-8 encoding
        nepali_stopwords = [line.strip() for line in f] # Indented this line by 4 spaces

    words = text.split()
    filtered_words = [word for word in words if word not in nepali_stopwords]
    filtered_text = ' '.join(filtered_words)
    return filtered_text


In [ ]:
from functools import lru_cache
import hashlib

@lru_cache(maxsize=128)
def calculate_embeddings(sentences):
    # Convert the list of sentences into a tuple to make it hashable
    sentences_tuple = tuple(sentences) # converting list to tuple to make it hashable for lru_cache

    model = SentenceTransformer('all-mpnet-base-v2')  # or a Nepali-specific model
    sentence_embeddings = model.encode(sentences_tuple)  # Use sentences_tuple here
    return sentence_embeddings

In [ ]:
def score_sentences_tfidf(sentences):
    vectorizer = TfidfVectorizer()
    vectorizer.fit(sentences)
    tfidf_vectors = vectorizer.transform(sentences)
    return tfidf_vectors.sum(axis=1).A1

def score_sentences_similarity(embeddings):
    centroid_embedding = np.mean(embeddings, axis=0)
    return cosine_similarity(embeddings, [centroid_embedding])

def score_sentences_textrank(embeddings):
    similarity_matrix = cosine_similarity(embeddings)
    nx_graph = nx.from_numpy_array(similarity_matrix)
    return nx.pagerank(nx_graph)

In [ ]:
def rank_array(arr):
    # If arr is a dictionary, convert its values to a list before flattening
    if isinstance(arr, dict):
        arr = list(arr.values())

    # Now, flatten the array (if it's a NumPy array or a nested list)
    arr = np.array(arr).flatten()
    temp = arr.argsort()
    ranks = np.empty_like(temp)
    ranks[temp] = np.arange(len(arr))
    return ranks

In [ ]:
def generate_summary(text, num_sentences_in_summary=None):
    cleaned_text = clean_nepali_text(text)
    filtered_text =remove_stopwords(cleaned_text)

    # Sentence Segmentation
    global sentences # Declare sentences as global to modify it within the function
    sentences = sentence_tokenize.sentence_split(filtered_text, lang='ne')

    # Create a hash of the sentences list
    sentences_hash = hashlib.sha256(str(sentences).encode()).hexdigest()

    # Sentence Embedding
    embeddings = calculate_embeddings(sentences_hash)

    # Sentence Scoring
    tfidf_scores = score_sentences_tfidf(sentences)
    similarity_scores = score_sentences_similarity(embeddings).flatten()
    textrank_scores = list(score_sentences_textrank(embeddings).values())

    # Ensure all score arrays have the same length as the number of sentences
    num_sentences = len(sentences)
    tfidf_scores = tfidf_scores[:num_sentences]
    similarity_scores = similarity_scores[:num_sentences]
    textrank_scores = textrank_scores[:num_sentences]

    # Normalize Scores
    normalized_tfidf = (tfidf_scores - np.min(tfidf_scores)) / (np.max(tfidf_scores) - np.min(tfidf_scores))
    normalized_similarity = (similarity_scores - np.min(similarity_scores)) / (np.max(similarity_scores) - np.min(similarity_scores))
    normalized_textrank = (np.array(textrank_scores) - np.min(np.array(textrank_scores))) / (np.max(np.array(textrank_scores)) - np.min(np.array(textrank_scores)))

    # Combine Scores
    tfidf_weight = 0.3
    similarity_weight = 0.4
    textrank_weight = 0.3
    combined_scores = tfidf_weight * normalized_tfidf + similarity_weight * normalized_similarity + textrank_weight * normalized_textrank

    # Sentence Selection
    total_sentences = len(sentences)

    if num_sentences_in_summary is None:
        num_sentences_in_summary = int(total_sentences * 0.3)  # Default to 30% if not specified
    elif isinstance(num_sentences_in_summary, float) and 0 < num_sentences_in_summary <= 1:
        num_sentences_in_summary = int(total_sentences * num_sentences_in_summary)  # Treat as a percentage
    elif num_sentences_in_summary > total_sentences:
        num_sentences_in_summary = total_sentences  # Cap at the total number of sentences

    # --- Sentence Selection and Return ---
    top_sentence_indices = np.argpartition(combined_scores, -num_sentences_in_summary)[-num_sentences_in_summary:]
    top_sentence_indices = sorted(top_sentence_indices)
    summary_sentences = [sentences[index] for index in top_sentence_indices]

    # Return Summary
    return ' '.join(summary_sentences)

In [ ]:
df.loc[10, 'formatted_article']

'"काठमाण्डाै – दुग्धजन्य पदार्थको लेबलमा म्याद नाघेको मिति उल्लेख नगर्ने एक डेरीलाई रु तीन लाख जरिवाना गरिएको छ । वाणिज्य आपूर्ति तथा उपभोक्ता संरक्षण विभागको बजार अनुगमन टोलीले सोमबार भक्तपुरमा एक डेरीलाई रु तीन लाख जरिवाना गरेको छ । अनुगमन टोलीले डेरीले लेबलिङ नियम पालना नगरेको र बजारमा वितरण का लागि नयाँ प्याकेजिङ मिति उल्लेख नगरी पुरानो दूध पुनः प्याकिङ गरेको पाइएको हो । उपभोक्ता संरक्षण ऐनअनुसार दूधजस्ता अत्यावश्यक पेय पदार्थको उत्पादन, परिमाण, गुणस्तर र अन्य सान्दर्भिक विवरणसहितको लेबल लगाउनु अनिवार्य छ । तर उल्लिखित डेरीले यी नियमको पालना नगरेको र लेबल बिनाको उक्त दूध समेत नष्ट गरिएको विभागको भनाइ छ ।"'

In [ ]:
nepali_text = "नेपाली साहित्य नेपाली भाषा, समाज, संस्कृति, र इतिहासको अभिव्यक्तिको सशक्त माध्यम हो, जसले मानव जीवनका विविध पक्षलाई उजागर गर्दै समाजमा चेतना फैलाउने महत्वपूर्ण भूमिका निर्वाह गरेको छ। यसको इतिहास प्राचीन कालदेखि नै मौखिक परम्परा, धार्मिक ग्रन्थ, र संस्कृत साहित्यसँग गहिरो रूपमा जोडिएको पाइन्छ, जहाँ लिच्छविकालदेखि नै विभिन्न काव्य, गद्य, र धार्मिक ग्रन्थहरूको निर्माण भएको देखिन्छ। मध्यकालमा नेपाल भाषामा लेखिएका साहित्यिक कृतिहरू प्रमुख थिए, भने भक्ति साहित्यको विकासले नेपाली भाषामा काव्य तथा अन्य विधाहरूको विस्तार गर्‍यो। यसै क्रममा, आदिकवि भानुभक्त आचार्यले संस्कृतमा रहेको रामायणलाई नेपाली भाषामा सरल रूपमा अनुवाद गरेर नेपाली साहित्यको लिखित परम्परालाई नयाँ दिशा दिए, जसले जनसाधारणले सजिलै बुझ्न सक्ने साहित्यिक संरचनाको सुरुवात गर्‍यो। त्यसपछि १९औं शताब्दीको उत्तरार्धमा मोतीराम भट्टले नेपाली गद्य तथा पद्य साहित्यको विकासमा महत्त्वपूर्ण योगदान दिँदै कविता, गजल, निबन्ध, तथा जीवनचरित्र लेखनलाई नयाँ उचाइमा पुर्याए। २०औं शताब्दीमा नेपाली साहित्य अझ व्यवस्थित र सशक्त हुँदै गयो, जहाँ लेखनाथ पौड्याल, लक्ष्मीप्रसाद देवकोटा, सिद्धिचरण श्रेष्ठ, माधव घिमिरे, भूपि शेरचन, बैरागी काइँला, र पारिजातजस्ता स्रष्टाहरूले नेपाली साहित्यलाई बहुआयामिक बनाउँदै विविध विधाहरूमा लेखन गरिरहेका थिए। विशेषगरी, महाकवि लक्ष्मीप्रसाद देवकोटाले नेपाली कवितामा गहिरो भावनात्मक अभिव्यक्ति, प्रकृतिप्रेम, दार्शनिकता, र मानवीय संवेदनशीलतालाई प्राथमिकता दिँदै कवितालाई एक विशिष्ट रूप प्रदान गरे, जसको उदाहरणस्वरूप ‘मुनामदन’ नेपाली साहित्यको सर्वाधिक लोकप्रिय खण्डकाव्य बनेको छ। यसैगरी, आधुनिक कविहरूले समाजका यथार्थपरक विषयवस्तुहरूलाई समेट्दै नेपाली साहित्यलाई विद्रोही स्वर, सामाजिक चेतना, र यथार्थवादी दृष्टिकोणतर्फ उन्मुख गराए, जसको उदाहरण भूपि शेरचनको ‘घुम्ने मेचमाथि अन्धो मान्छे’ कवितासंग्रहमा पाइन्छ। नेपाली साहित्यमा विभिन्न विधाहरू रहेका छन्, जसमा कविता, गजल, कथा, उपन्यास, निबन्ध, नाटक, र यात्रासाहित्य प्रमुख मानिन्छन्, जहाँ कविताले प्रेम, प्रकृति, देशभक्ति, विद्रोह, र मानवता जस्ता विषयहरूलाई उजागर गर्छ भने गजल विशेष गरी प्रेम, पीडा, र जीवनका गूढ पक्षहरूलाई गहिरो रूपमा प्रस्तुत गर्छ। नेपाली कथाले समाजको यथार्थ चित्रण गर्दै वर्गसंघर्ष, गरिबी, महिला अधिकार, जातीयता, समाजमा भएका विभिन्न विकृति–विसंगतिहरू, तथा परिवर्तनका कथाहरू प्रस्तुत गर्छ, जसको उदाहरण पारिजातको ‘शिरिषको फूल’ उपन्यासमा देख्न सकिन्छ। नेपाली साहित्यको निबन्ध र यात्रासाहित्यले विचारप्रधान तथा अनुभवजन्य विषयवस्तुहरू प्रस्तुत गर्दै पाठकलाई गहिरो चिन्तन गर्न प्रेरित गर्छ, जसमा बुद्धिसागरका समकालीन निबन्धहरू लोकप्रिय छन्। प्रविधिको विकाससँगै नेपाली साहित्य डिजिटल माध्यममा पनि प्रवाहित भइरहेको छ, जसले अनलाइन ब्लग, ई-पत्रिका, तथा सामाजिक सञ्जालहरूमा नेपाली साहित्यलाई विश्वभर फैलाउन महत्त्वपूर्ण भूमिका खेलेको छ, र यसले नेपाली लेखकहरूलाई अन्तर्राष्ट्रिय रूपमा चिनाउने अवसर प्रदान गरेको छ। आजको पुस्तामा आधुनिक साहित्यकारहरू समसामयिक विषयमा लेख्न सक्रिय छन्, जसले नेपाली समाजमा रहेका राजनीतिक, सामाजिक, तथा सांस्कृतिक विषयहरूलाई समेट्दै लेखनलाई अझ सशक्त बनाइरहेका छन्, जसका कारण नेपाली साहित्यको प्रभाव निरन्तर बढ्दै गइरहेको छ। समग्रमा, नेपाली साहित्यले समाजका विविध पक्षलाई उजागर गर्दै, चेतना फैलाउने, मनोरञ्जन प्रदान गर्ने, तथा मानवीय संवेदनशीलतालाई अभिव्यक्त गर्ने महत्त्वपूर्ण भूमिका निर्वाह गर्दै आएको छ, जसले नेपाली भाषा र संस्कृतिको समृद्धिमा महत्त्वपूर्ण योगदान पुर्याइरहेको छ।"
summary = generate_summary(nepali_text)
print(summary)

विशेषगरी महाकवि लक्ष्मीप्रसाद देवकोटाले नेपाली कवितामा गहिरो भावनात्मक अभिव्यक्ति प्रकृतिप्रेम दार्शनिकता मानवीय संवेदनशीलतालाई प्राथमिकता दिँदै कवितालाई विशिष्ट प्रदान गरे उदाहरणस्वरूप मुनामदन नेपाली साहित्यको सर्वाधिक लोकप्रिय खण्डकाव्य बनेको छ। नेपाली साहित्यमा विभिन्न विधाहरू कविता गजल कथा उपन्यास निबन्ध नाटक यात्रासाहित्य प्रमुख मानिन्छन् कविताले प्रेम प्रकृति देशभक्ति विद्रोह मानवता जस्ता विषयहरूलाई उजागर गजल प्रेम पीडा जीवनका गूढ पक्षहरूलाई गहिरो रूपमा प्रस्तुत गर्छ। प्रविधिको विकाससँगै नेपाली साहित्य डिजिटल माध्यममा प्रवाहित भइरहेको अनलाइन ब्लग ईपत्रिका सामाजिक सञ्जालहरूमा नेपाली साहित्यलाई विश्वभर फैलाउन महत्त्वपूर्ण भूमिका खेलेको यसले नेपाली लेखकहरूलाई अन्तर्राष्ट्रिय रूपमा चिनाउने अवसर प्रदान छ। पुस्तामा आधुनिक साहित्यकारहरू समसामयिक विषयमा लेख्न सक्रिय नेपाली समाजमा राजनीतिक सामाजिक सांस्कृतिक विषयहरूलाई समेट्दै लेखनलाई अझ सशक्त बनाइरहेका जसका कारण नेपाली साहित्यको प्रभाव निरन्तर बढ्दै गइरहेको छ।


In [ ]:
nepali_text

'नेपाली साहित्य नेपाली भाषा, समाज, संस्कृति, र इतिहासको अभिव्यक्तिको सशक्त माध्यम हो, जसले मानव जीवनका विविध पक्षलाई उजागर गर्दै समाजमा चेतना फैलाउने महत्वपूर्ण भूमिका निर्वाह गरेको छ। यसको इतिहास प्राचीन कालदेखि नै मौखिक परम्परा, धार्मिक ग्रन्थ, र संस्कृत साहित्यसँग गहिरो रूपमा जोडिएको पाइन्छ, जहाँ लिच्छविकालदेखि नै विभिन्न काव्य, गद्य, र धार्मिक ग्रन्थहरूको निर्माण भएको देखिन्छ। मध्यकालमा नेपाल भाषामा लेखिएका साहित्यिक कृतिहरू प्रमुख थिए, भने भक्ति साहित्यको विकासले नेपाली भाषामा काव्य तथा अन्य विधाहरूको विस्तार गर्\u200dयो। यसै क्रममा, आदिकवि भानुभक्त आचार्यले संस्कृतमा रहेको रामायणलाई नेपाली भाषामा सरल रूपमा अनुवाद गरेर नेपाली साहित्यको लिखित परम्परालाई नयाँ दिशा दिए, जसले जनसाधारणले सजिलै बुझ्न सक्ने साहित्यिक संरचनाको सुरुवात गर्\u200dयो। त्यसपछि १९औं शताब्दीको उत्तरार्धमा मोतीराम भट्टले नेपाली गद्य तथा पद्य साहित्यको विकासमा महत्त्वपूर्ण योगदान दिँदै कविता, गजल, निबन्ध, तथा जीवनचरित्र लेखनलाई नयाँ उचाइमा पुर्याए। २०औं शताब्दीमा नेपाली साहित्य अझ व्यवस्थित र सशक्त हुँदै गयो, जहाँ

In [ ]:
summary

'विशेषगरी महाकवि लक्ष्मीप्रसाद देवकोटाले नेपाली कवितामा गहिरो भावनात्मक अभिव्यक्ति प्रकृतिप्रेम दार्शनिकता मानवीय संवेदनशीलतालाई प्राथमिकता दिँदै कवितालाई विशिष्ट प्रदान गरे उदाहरणस्वरूप मुनामदन नेपाली साहित्यको सर्वाधिक लोकप्रिय खण्डकाव्य बनेको छ। नेपाली साहित्यमा विभिन्न विधाहरू कविता गजल कथा उपन्यास निबन्ध नाटक यात्रासाहित्य प्रमुख मानिन्छन् कविताले प्रेम प्रकृति देशभक्ति विद्रोह मानवता जस्ता विषयहरूलाई उजागर गजल प्रेम पीडा जीवनका गूढ पक्षहरूलाई गहिरो रूपमा प्रस्तुत गर्छ। प्रविधिको विकाससँगै नेपाली साहित्य डिजिटल माध्यममा प्रवाहित भइरहेको अनलाइन ब्लग ईपत्रिका सामाजिक सञ्जालहरूमा नेपाली साहित्यलाई विश्वभर फैलाउन महत्त्वपूर्ण भूमिका खेलेको यसले नेपाली लेखकहरूलाई अन्तर्राष्ट्रिय रूपमा चिनाउने अवसर प्रदान छ। पुस्तामा आधुनिक साहित्यकारहरू समसामयिक विषयमा लेख्न सक्रिय नेपाली समाजमा राजनीतिक सामाजिक सांस्कृतिक विषयहरूलाई समेट्दै लेखनलाई अझ सशक्त बनाइरहेका जसका कारण नेपाली साहित्यको प्रभाव निरन्तर बढ्दै गइरहेको छ।'

In [ ]:
nepali_text = "काठमाडौँ — आगोको लप्काले बेलुन फुट्दा घाइते भएका उपप्रधान तथा अर्थमन्त्री विष्णुप्रसाद पौडेल र पोखरा महानगरपालिकाका प्रमुख धनराज आचार्यको कीर्तिपुर अस्पतालमा उपचार सुरु भएको छ ।अर्थमन्त्री पौडेल र मेयर आचार्यलाई सघन कक्ष (आईसीयू)मा भर्ना गरेर उपचार गरिरहेको अस्पतालका निर्देशक किरणकिशोर नकर्मीले विज्ञप्तिमार्फत जानकारी दिएका हुन् ।'आज २०८१ फागुन ३ गते शनिबार पोखराबाट जलन घाइते भई यस कीर्तिपुर अस्पतालमा दिउँसोको ३:३० बजे उपप्रधान तथा अर्थमन्त्री विष्णु पौडेल तथा पोखरा महानगरपालिकाका मेयर धनराज आचार्य उपचारार्थ सघन उपचार कक्षमा भर्ना हुनुभएको छ,' विज्ञप्तिमा लेखिएको छ, 'उहाँहरूको प्रारम्भिक मूल्यांकनअनुसार ५-६% जलेको तर अनुहारमा जलन भएकाले श्वासनली र फोक्सोमा असर पर्न सक्ने सम्भावना रहेकाले उच्च निगरानीमा उपचार गर्नुपर्ने आवश्यकता देखिन्छ ।'डा. विशाल कार्कीले दुवैको अवस्था सामान्य रहेको भन्दै पोलेको घाउ गहिरोमा परिणत हुने खतरा रहेकाले आईसीयूमा राखेर उपचार थालिएको जनाए । 'उहाँहरुको अवस्था सामान्य नै छ । पोलेको घाउ गहिरो देखिएको छैन । तर, गहिरोमा परिणत हुने खतरा हुन्छ । पड्किँदा उहाँहरूले धूवाँ निल्नु भयो । त्यसले भित्र असर परेको छ कि भन्ने अब्जर्ब गर्नका लागि पहिलो ४८ घण्टा आईसीयूमा राख्ने प्लान गरेका छौं । घाउ गहिरो हुने/नहुने ३/४ दिनमा थाहा हुन्छ । त्यो बेलासम्म ड्रेसिङ गर्ने, औषधि दिने गर्छौं,' उनले कान्तिपुरसँग भने ।उनीहरूको उपचारमा प्रा डा बीडी झा, डा. प्रवीण गिरी लगायतको सघन उपचार कक्षको टोली, प्रा. डा. शंकरमान राई, डा. विशाल कार्की, डा. मनीष यादवलगायतको जलन विशेषज्ञहरू, डा. प्रयुस अर्याल लगायत आकस्मिक उपचार टोली र रिना गौचन, निर्मला कुँवर, तृष्णा महर्जन लगायत नर्सिङ टोली संलग्न रहेको विज्ञप्तिमा उल्लेख छ ।पोखरा भ्रमण वर्ष, २०२५ को उद्घाटनका क्रममा शनिबार रिमोर्टबाट दीप प्रज्वलनपछि बेलुन उडाउने क्रममा आगोको लप्काले बेलुन पड्किएको थियो । आगोका झिल्का उनीहरुको शरीरमा लागेको थियो । दुवै जनालाई पोखरामा प्राथमिक उपचारपछि कीर्तिपुर अस्पतालमा ल्याइएको हो ।"
summary = generate_summary(nepali_text)
print(summary)

आज 2081 फागुन 3 गते शनिबार पोखराबाट जलन घाइते भई कीर्तिपुर अस्पतालमा दिउँसोको 330 बजे उपप्रधान अर्थमन्त्री विष्णु पौडेल पोखरा महानगरपालिकाका मेयर धनराज आचार्य उपचारार्थ सघन उपचार कक्षमा भर्ना हुनुभएको विज्ञप्तिमा लेखिएको उहाँहरूको प्रारम्भिक मूल्यांकनअनुसार 56 जलेको अनुहारमा जलन भएकाले श्वासनली फोक्सोमा असर पर्न सक्ने सम्भावना रहेकाले उच्च निगरानीमा उपचार गर्नुपर्ने आवश्यकता । उहाँहरुको अवस्था सामान्य । पड्किँदा उहाँहरूले धूवाँ निल्नु भयो । उनीहरूको उपचारमा प्रा डा बीडी झा डा प्रवीण गिरी लगायतको सघन उपचार कक्षको टोली प्रा डा शंकरमान राई डा विशाल कार्की डा मनीष यादवलगायतको जलन विशेषज्ञहरू डा प्रयुस अर्याल लगायत आकस्मिक उपचार टोली रिना गौचन निर्मला कुँवर तृष्णा महर्जन लगायत नर्सिङ टोली संलग्न विज्ञप्तिमा उल्लेख ।


In [ ]:
nepali_text

"काठमाडौँ — आगोको लप्काले बेलुन फुट्दा घाइते भएका उपप्रधान तथा अर्थमन्त्री विष्णुप्रसाद पौडेल र पोखरा महानगरपालिकाका प्रमुख धनराज आचार्यको कीर्तिपुर अस्पतालमा उपचार सुरु भएको छ ।अर्थमन्त्री पौडेल र मेयर आचार्यलाई सघन कक्ष (आईसीयू)मा भर्ना गरेर उपचार गरिरहेको अस्पतालका निर्देशक किरणकिशोर नकर्मीले विज्ञप्तिमार्फत जानकारी दिएका हुन् ।'आज २०८१ फागुन ३ गते शनिबार पोखराबाट जलन घाइते भई यस कीर्तिपुर अस्पतालमा दिउँसोको ३:३० बजे उपप्रधान तथा अर्थमन्त्री विष्णु पौडेल तथा पोखरा महानगरपालिकाका मेयर धनराज आचार्य उपचारार्थ सघन उपचार कक्षमा भर्ना हुनुभएको छ,' विज्ञप्तिमा लेखिएको छ, 'उहाँहरूको प्रारम्भिक मूल्यांकनअनुसार ५-६% जलेको तर अनुहारमा जलन भएकाले श्वासनली र फोक्सोमा असर पर्न सक्ने सम्भावना रहेकाले उच्च निगरानीमा उपचार गर्नुपर्ने आवश्यकता देखिन्छ ।'डा. विशाल कार्कीले दुवैको अवस्था सामान्य रहेको भन्दै पोलेको घाउ गहिरोमा परिणत हुने खतरा रहेकाले आईसीयूमा राखेर उपचार थालिएको जनाए । 'उहाँहरुको अवस्था सामान्य नै छ । पोलेको घाउ गहिरो देखिएको छैन । तर, गहिरोमा परिणत हुने खतरा हुन्छ । पड्किँदा उहाँहरूले 

In [ ]:
summary

'आज 2081 फागुन 3 गते शनिबार पोखराबाट जलन घाइते भई कीर्तिपुर अस्पतालमा दिउँसोको 330 बजे उपप्रधान अर्थमन्त्री विष्णु पौडेल पोखरा महानगरपालिकाका मेयर धनराज आचार्य उपचारार्थ सघन उपचार कक्षमा भर्ना हुनुभएको विज्ञप्तिमा लेखिएको उहाँहरूको प्रारम्भिक मूल्यांकनअनुसार 56 जलेको अनुहारमा जलन भएकाले श्वासनली फोक्सोमा असर पर्न सक्ने सम्भावना रहेकाले उच्च निगरानीमा उपचार गर्नुपर्ने आवश्यकता । उहाँहरुको अवस्था सामान्य । पड्किँदा उहाँहरूले धूवाँ निल्नु भयो । उनीहरूको उपचारमा प्रा डा बीडी झा डा प्रवीण गिरी लगायतको सघन उपचार कक्षको टोली प्रा डा शंकरमान राई डा विशाल कार्की डा मनीष यादवलगायतको जलन विशेषज्ञहरू डा प्रयुस अर्याल लगायत आकस्मिक उपचार टोली रिना गौचन निर्मला कुँवर तृष्णा महर्जन लगायत नर्सिङ टोली संलग्न विज्ञप्तिमा उल्लेख ।'